# BL00 — exact Epoch-33 G4 restoration to continuous training from Epoch 34

The latest reported BL00 session completed Epochs 18–33, with 7,972 optimizer updates and B-VAL mAP50–95 0.37353. The externally recorded latest manifest SHA256 is `c06a08777fd31072c004004d4d924008047f2fafb0be3321a98ed5ac1f84aa29`. This notebook independently verifies that manifest, its referenced checkpoint files, and the two rank states before training.

## Required Kaggle input
Attach the latest saved notebook output at `/kaggle/input/notebooks/hebut123/grazpedwri-dx-bl00-ddp-b16`. It must contain `BL00_CONTINUOUS_B16_OUTPUT/g4/BL00_EPOCH033_two_rank_manifest.json` and the matching rank0/rank1 `.pt`, best checkpoint, `LATEST_MANIFEST.json` and `BL00_CONTINUOUS_B16_OUTPUT/bootstrap` inputs. The earlier Epoch-17 output is not suitable. Attach the same four BL00 scientific datasets as the previous session. GPU: 2×T4; Internet: OFF.

## Training protocol
Resume Epoch **34 of 100**, native full-state restore, original standard YOLO11s BL00 baseline, frozen scientific source commit `f440d329d4544ec08ad63c1972bcf9a8b8a7eed8`, Split-B TRAIN/B-VAL, DDP total batch16, imgsz1024, SGD, AMP. No MV00/MV01 checkpoint or AMP override. B-TEST SEALED. Each session stops only after a completed-epoch two-rank G4, bounded to at most 16 new epochs or 7.5h. Epoch 100 is represented by the final two-rank G4 and B-VAL metrics; an MV01-style `TERMINAL.json` is not expected.

## Run and verify
Save Version → Save & Run All. The required preflight output is `AUTHENTICATED G4`, `COMPLETED EPOCHS: 33 NEXT: 34`, `RESTORE OPTIMIZER UPDATES: 7972`, and `BL00_RESTRICTED_G4_DESERIALIZATION: TWO_RANK_CPU_PREFLIGHT_PASS`. After completion inspect `BL00 CONTINUATION SESSION: PASS`, the latest 2-rank G4 SHA256, and the progress. Never assume a visible Epoch-34 batch proves the full epoch checkpoint has been saved.


In [ ]:
# BL00 - exact two-rank G4 restoration, Epoch 34 onward (Kaggle, Internet OFF)
# Source: authenticated BL00 Epoch-33 saved-version output and frozen GRAZPEDWRI Split B.
# Runs at most 16 epochs/session, with an authentic two-rank G4 after EACH epoch.
# On next saved-version session, point PRIOR_NOTEBOOK_OUTPUT at that output and rerun.
from __future__ import annotations

import ast
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

PRIOR_NOTEBOOK_OUTPUT = Path(
    "/kaggle/input/notebooks/hebut123/grazpedwri-dx-bl00-ddp-b16"
)
EPOCH1_MANIFEST_SHA256 = (
    "b1d1febe465a87445a99ab4527f7a5a26b74f821005fa0b064aa58c150214cc8"
)
EXPECTED_PREVIOUS_COMPLETED_EPOCHS = 33
EXPECTED_PREVIOUS_OPTIMIZER_UPDATES = 7972
EXPECTED_PREVIOUS_MANIFEST_SHA256 = (
    "c06a08777fd31072c004004d4d924008047f2fafb0be3321a98ed5ac1f84aa29"
)
FROZEN_COMMIT = "f440d329d4544ec08ad63c1972bcf9a8b8a7eed8"
OUTPUT = Path("/kaggle/working/BL00_CONTINUOUS_B16_OUTPUT")
BOOTSTRAP = OUTPUT / "bootstrap"
MAX_EPOCHS_THIS_SESSION = 16
MAX_WALL_HOURS_THIS_SESSION = 7.5


def need(condition, code):
    if not condition:
        raise RuntimeError("BL00_CONT_" + code)


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for part in iter(lambda: f.read(1 << 20), b""):
            h.update(part)
    return h.hexdigest()


def safe_regular(path):
    path = Path(path)
    return path.is_file() and not path.is_symlink()


def load_py(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    need(spec is not None and spec.loader is not None, "IMPORT_SPEC")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


# -------- New continuation modules (not modifications to frozen source) --------
MODULES = {}

MODULES["bl00_continue_rank0.py"] = r'''
"""Read the accepted rank0 validation boundary; only broaden its epoch guard."""
from bl00_epoch1_rank0 import validate_epoch1 as _original
# Runtime uses a directly patched copy of the accepted function below.
'''

MODULES["bl00_continue_trainer.py"] = r'''
"""BL00 full-state resumption from an independently SHA-authenticated G4."""
from __future__ import annotations
import math
import os
import random
import shutil
from pathlib import Path
import numpy as np
import torch
import torch.distributed as dist

from ultralytics.utils.torch_utils import unwrap_model
from ultralytics.models.yolo.detect.d4e_transform_rng_state import (
    capture_augmentation_rng, restore_augmentation_rng
)
from bl00_epoch1_trainer import BL00Epoch1Trainer


def require(ok, tag):
    if not ok:
        raise RuntimeError("BL00_CONT_RESTORE_" + tag)


class BL00ContinueTrainer(BL00Epoch1Trainer):
    def __init__(self, *, previous_g4, previous_manifest, **kwargs):
        self._bl00_previous_g4 = Path(previous_g4)
        self._bl00_previous_manifest = previous_manifest
        self._bl00_resume_loaded = False
        self._bl00_resume_last_opt_step = None
        super().__init__(**kwargs)

    def _setup_train(self):
        super()._setup_train()
        rank = dist.get_rank()
        manifest = self._bl00_previous_manifest
        completed = int(manifest["completed_epochs"])
        require(1 <= completed < 100, "EPOCH_RANGE")
        require(manifest["next_epoch_index"] == completed, "NEXT_INDEX")
        filename = f"BL00_EPOCH{completed:03d}_rank{rank}.pt"
        row = manifest["ranks"][str(rank)]
        require(row["filename"] == filename, "RANK_FILENAME")
        path = self._bl00_previous_g4 / filename
        # SHA was checked in parent before torchrun; check again here.
        from bl00_continue_g4 import sha
        require(sha(path) == row["sha256"], "RANK_SHA")
        from bl00_continue_safe_load import load_authenticated_rank_state
        state = load_authenticated_rank_state(path, row["sha256"])
        require(state["arm"] == "BL00" and state["world_size"] == 2
                and state["rank"] == rank
                and state["completed_epoch_index"] + 1 == completed
                and state["next_epoch_index"] == completed, "STATE_IDENTITY")
        require(state["b_test_access"] == "NONE", "TEST_ACCESS")
        require(state["optimizer_updates"] == manifest["optimizer_updates"]
                and state["ema_updates"] == manifest["optimizer_updates"], "UPDATE_IDENTITY")
        require(state["last_opt_step"] == manifest["last_opt_step"]
                and state["accumulate"] == manifest["accumulate"], "ACCUMULATION_IDENTITY")
        require(state["sampler_next_epoch"] == completed, "SAMPLER_INDEX")
        require(float(state["best_fitness"]) == float(manifest["best_fitness"]),
                "FITNESS_IDENTITY")

        model = unwrap_model(self.model)
        require(set(model.state_dict()) == set(state["model"]), "MODEL_KEY_SET")
        model.load_state_dict(state["model"], strict=True)
        self.ema.ema.load_state_dict(state["ema"], strict=True)
        self.optimizer.load_state_dict(state["optimizer"])
        self.scheduler.load_state_dict(state["scheduler"])
        self.scaler.load_state_dict(state["scaler"])
        # Verify restored trainable/model/EMA/SGD state before even one update.
        for key, original in state["model"].items():
            require(torch.equal(model.state_dict()[key].detach().cpu(), original),
                    "MODEL_ZERO_UPDATE_" + key)
        for key, original in state["ema"].items():
            require(torch.equal(self.ema.ema.state_dict()[key].detach().cpu(), original),
                    "EMA_ZERO_UPDATE_" + key)
        actual_opt = self.optimizer.state_dict()
        expected_opt = state["optimizer"]
        require(actual_opt["param_groups"] == expected_opt["param_groups"],
                "OPTIMIZER_GROUP_RESTORE")
        require(set(actual_opt["state"]) == set(expected_opt["state"]),
                "OPTIMIZER_STATE_KEY_RESTORE")
        for idx, saved_fields in expected_opt["state"].items():
            current_fields = actual_opt["state"][idx]
            require(set(current_fields) == set(saved_fields),
                    "OPTIMIZER_FIELD_KEYS")
            for field, original in saved_fields.items():
                restored = current_fields[field]
                if isinstance(original, torch.Tensor):
                    require(torch.equal(restored.detach().cpu(), original),
                            "OPTIMIZER_TENSOR_ZERO_UPDATE")
                else:
                    require(restored == original, "OPTIMIZER_VALUE_ZERO_UPDATE")
        require(self.scaler.state_dict() == state["scaler"], "SCALER_RESTORE")
        require(self.scheduler.state_dict() == state["scheduler"],
                "SCHEDULER_RESTORE")
        self.ema.updates = int(state["ema_updates"])
        self._bl00_optimizer_updates = int(state["optimizer_updates"])
        self._bl00_resume_last_opt_step = int(state["last_opt_step"])
        self.accumulate = int(state["accumulate"])
        self.start_epoch = completed
        self.best_fitness = float(state["best_fitness"])
        self.fitness = float(state["fitness"])
        self.metrics = {str(k):float(v) for k,v in state["metrics"].items()}
        self.stopper.best_fitness = self.best_fitness
        self.stopper.best_epoch = int(state["stopper"]["best_epoch"])
        self.stopper.possible_stop = bool(state["stopper"]["possible_stop"])

        param = dict(model.named_parameters())
        require(set(param) == set(state["pending_gradients"]), "GRAD_KEY_SET")
        for name, p in param.items():
            saved = state["pending_gradients"][name]
            if saved is None:
                p.grad = None
            else:
                require(saved.shape == p.shape, "GRAD_SHAPE_" + name)
                p.grad = saved.to(device=p.device, dtype=p.dtype).clone()

        self.train_loader.vgra_pair_batch_sampler.set_epoch(completed)
        self.train_loader.generator.set_state(state["train_loader_generator_rng"])
        self.test_loader.generator.set_state(state["val_loader_generator_rng"])
        restore_augmentation_rng(
            self.train_loader.dataset.transforms,
            state["augmentation_rng"],
        )
        require(capture_augmentation_rng(
            self.train_loader.dataset.transforms
        ) == state["augmentation_rng"], "AUGMENT_RNG_ROUNDTRIP")

        # The native trainer must not zero restored pending gradients.
        self._bl00_resume_loaded = True

        # Copy the historically best native checkpoint, even if the next
        # validation does not improve on the previous epoch's fitness.
        if rank == 0:
            oldbest = manifest.get("best_filename", "best.pt")
            best_source = self._bl00_previous_g4 / oldbest
            require(sha(best_source) == manifest["selected_best_sha256"],
                    "OLD_BEST_SHA")
            require(not self.best.exists(), "NEW_BEST_ALREADY_EXISTS")
            shutil.copyfile(best_source, self.best)
        dist.barrier()

        # Restore global RNG LAST: model/dataloader/optimizer creation and
        # verification above must not perturb the epoch-2 random stream.
        nprng = state["numpy_rng"]
        np.random.set_state((
            nprng["method"], np.array(nprng["state"],dtype=np.uint32),
            int(nprng["pos"]), int(nprng["has_gauss"]),
            float(nprng["gauss"])
        ))
        random.setstate(state["python_rng"])
        torch.set_rng_state(state["torch_rng"])
        torch.cuda.set_rng_state(state["cuda_rng"], device=self.device)
        self._bl00_session_start = __import__("time").monotonic()
        self._bl00_session_first_epoch_index = completed
        self._bl00_last_accepted_epoch = completed
        dist.barrier()
        print(f"BL00_RESTORED_RANK_{rank}: next_epoch={completed+1},"
              f" optimizer_updates={self._bl00_optimizer_updates},"
              f" last_opt_step={self._bl00_resume_last_opt_step}",
              flush=True)

    def optimizer_step(self):
        # Same native scaler->unscale->clip->SGD->scaler->EMA order as the
        # accepted Epoch-1 trainer, without its epoch==0 restriction.
        require(dist.is_initialized() and 1 <= self.epoch < 100, "OPTIMIZER_EPOCH")
        before = float(self.scaler.get_scale())
        self.scaler.unscale_(self.optimizer)
        norm = torch.nn.utils.clip_grad_norm_(
            self.model.parameters(), max_norm=10.0,
            error_if_nonfinite=True
        )
        require(bool(torch.isfinite(norm).item()), "NONFINITE_GRAD")
        self.scaler.step(self.optimizer)
        self.scaler.update()
        require(float(self.scaler.get_scale()) >= before, "AMP_SKIPPED")
        self.optimizer.zero_grad()
        self.ema.update(self.model)
        self._bl00_optimizer_updates += 1

    def final_eval(self):
        raise RuntimeError("BL00_CONT_FINAL_EVAL_MUST_USE_GOVERNED_BVAL")
'''

MODULES["bl00_continue_g4.py"] = r'''
"""Portable two-rank G4 after every completed BL00 epoch, with safe stop."""
from __future__ import annotations
import hashlib
import json
import math
import os
import random
import shutil
import time
from pathlib import Path
import numpy as np
import torch
import torch.distributed as dist
from ultralytics.utils.torch_utils import unwrap_model
from ultralytics.models.yolo.detect.d4e_transform_rng_state import capture_augmentation_rng
from bl00_epoch1_g4 import cpu_tree, py_np_rng, write_rank_file, sha

class BL00ContinuationSessionComplete(Exception):
    """Success: completed epoch and committed a two-rank G4."""


def require(ok, tag):
    if not ok:
        raise RuntimeError("BL00_CONT_G4_" + tag)


def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_name(path.name + ".writing")
    require(not tmp.exists(), "JSON_TEMP_EXISTS")
    with tmp.open("x",encoding="utf-8") as f:
        json.dump(obj,f,sort_keys=True,indent=2,allow_nan=False)
        f.write("\n")
    os.replace(tmp,path)


def capture_and_maybe_stop(trainer, epoch, nb, last_opt_step):
    require(dist.is_initialized() and dist.get_world_size()==2
            and dist.get_backend()=="nccl", "TWO_RANK_NCCL")
    require(1 <= epoch < 100 and epoch==trainer.epoch, "EPOCH")
    require(nb==890 and len(trainer.train_loader)==890
            and len(trainer.train_loader.dataset)==14227
            and len(trainer.test_loader.dataset)==3049, "POPULATION")
    require(trainer.train_loader.vgra_pair_batch_sampler.epoch==epoch+1,
            "SAMPLER_NEXT_EPOCH")
    require(trainer.ema.updates==trainer._bl00_optimizer_updates,
            "EMA_UPDATES")
    require(trainer._bl00_optimizer_updates >
            int(trainer._bl00_previous_manifest["optimizer_updates"]),
            "OPTIMIZER_STALLED")
    require(trainer.fitness is not None
            and math.isfinite(float(trainer.fitness))
            and float(trainer.metrics.get("d4_fixed_conf_025_eval_images",0))==3049
            and "metrics/mAP50-95(B)" in trainer.metrics, "VALIDATION")
    rank = dist.get_rank()
    completed = epoch+1
    work = Path(os.environ["BL00_CONT_OUTPUT"])
    g4 = work/"g4"
    require(g4.is_dir(), "G4_DIR")
    model = unwrap_model(trainer.model)
    state={
        "schema":"BL00_CONT_TWO_RANK_NATIVE_STATE_V1",
        "arm":"BL00","rank":rank,"world_size":2,
        "completed_epoch_index":epoch,"next_epoch_index":completed,
        "optimizer_updates":int(trainer._bl00_optimizer_updates),
        "last_opt_step":int(last_opt_step),
        "accumulate":int(trainer.accumulate),
        "ema_updates":int(trainer.ema.updates),
        "model":cpu_tree(model.state_dict()),
        "ema":cpu_tree(trainer.ema.ema.state_dict()),
        "optimizer":cpu_tree(trainer.optimizer.state_dict()),
        "scheduler":cpu_tree(trainer.scheduler.state_dict()),
        "scaler":cpu_tree(trainer.scaler.state_dict()),
        "pending_gradients":{k:cpu_tree(p.grad) for k,p in model.named_parameters()},
        "torch_rng":torch.get_rng_state().cpu(),
        "cuda_rng":torch.cuda.get_rng_state(trainer.device).cpu(),
        "numpy_rng":py_np_rng(),
        "python_rng":random.getstate(),
        "augmentation_rng":capture_augmentation_rng(trainer.train_loader.dataset.transforms),
        "metrics":{str(k):float(v) for k,v in trainer.metrics.items()},
        "fitness":float(trainer.fitness),
        "best_fitness":float(trainer.best_fitness),
        "stopper":{"best_epoch":int(trainer.stopper.best_epoch),
                   "possible_stop":bool(trainer.stopper.possible_stop)},
        "sampler_next_epoch":completed,
        "train_loader_generator_rng":trainer.train_loader.generator.get_state().cpu(),
        "val_loader_generator_rng":trainer.test_loader.generator.get_state().cpu(),
        "b_test_access":"NONE",
    }
    prefix=f"BL00_EPOCH{completed:03d}"
    filename=f"{prefix}_rank{rank}.pt"
    dest=g4/filename
    digest=write_rank_file(dest,state)
    local={
        "rank":rank,"filename":filename,"sha256":digest,
        "bytes":dest.stat().st_size,
        "optimizer_updates":state["optimizer_updates"],
        "last_opt_step":state["last_opt_step"],
        "accumulate":state["accumulate"],
        "ema_updates":state["ema_updates"],
        "fitness":state["fitness"],
        "sampler_next_epoch":completed,
    }
    gathered=[None,None]
    dist.all_gather_object(gathered,local)
    require([x["rank"] for x in gathered]==[0,1],"RANK_ORDER")
    for k in ("optimizer_updates","last_opt_step","accumulate",
              "ema_updates","fitness","sampler_next_epoch"):
        require(gathered[0][k]==gathered[1][k],"RANK_MISMATCH_"+k)
    dist.barrier()
    wire=[None]
    if rank==0:
        try:
            native_best=Path(trainer.best)
            native_last=Path(trainer.last)
            require(native_best.is_file() and native_last.is_file(),
                    "NATIVE_BEST_LAST")
            best_filename=f"{prefix}_best.pt"
            best_copy=g4/best_filename
            require(not best_copy.exists(), "BEST_ALREADY_EXISTS")
            shutil.copyfile(native_best,best_copy)
            best_sha=sha(best_copy)
            manifest={
                "schema":"BL00_CONT_COMPLETE_G4_MANIFEST_V1",
                "complete":True,"arm":"BL00","world_size":2,
                "frozen_commit":"f440d329d4544ec08ad63c1972bcf9a8b8a7eed8",
                "official_checkpoint_sha256":"85a76fe86dd8afe384648546b56a7a78580c7cb7b404fc595f97969322d502d5",
                "gate2b_receipt_sha256":"010ff19e06525598f264bfa8eddfd46cf0ba2e74bb9742e67f4cb12c3e7f0ae6",
                "gate3b_receipt_sha256":"19d76655cb047727b91372be8f5eff384fa0c32d4653719acfb324bcd9d5b2da",
                "assignment_sha256":"87aa5e9089c6282df14ae32917946d70d9f56eaa9a5d63443985877ab9dd1ef4",
                "completed_epochs":completed,"next_epoch_index":completed,
                "steps_per_rank":890,"train_images":14227,
                "operational_val_images":3049,
                "optimizer_updates":state["optimizer_updates"],
                "last_opt_step":state["last_opt_step"],
                "accumulate":state["accumulate"],
                "best_fitness":state["best_fitness"],
                "val_metrics":state["metrics"],
                "selected_best_sha256":best_sha,
                "best_filename":best_filename,
                "files":{**{x["filename"]:x["sha256"] for x in gathered},
                         best_filename:best_sha},
                "ranks":{"0":gathered[0],"1":gathered[1]},
                "native_last_sha256":sha(native_last),
                "b_test_access":"NONE",
            }
            name=f"{prefix}_two_rank_manifest.json"
            path=g4/name
            require(not path.exists(), "MANIFEST_ALREADY_EXISTS")
            atomic_json(path,manifest)
            mh=sha(path)
            latest={
                "schema":"BL00_G4_LATEST_POINTER_V1",
                "manifest":name,
                "manifest_sha256":mh,
                "completed_epochs":completed,
            }
            atomic_json(g4/"LATEST_MANIFEST.json",latest)
            wire[0]={"ok":True,"manifest_sha256":mh,
                     "manifest_filename":name,
                     "completed_epochs":completed}
            # Keep current+previous epoch; older epoch capsules may be
            # deleted only AFTER the new manifest/pointer is committed.
            old=completed-2
            if old>=2:
                for fn in (f"BL00_EPOCH{old:03d}_rank0.pt",
                           f"BL00_EPOCH{old:03d}_rank1.pt",
                           f"BL00_EPOCH{old:03d}_best.pt",
                           f"BL00_EPOCH{old:03d}_two_rank_manifest.json"):
                    (g4/fn).unlink(missing_ok=True)
        except Exception as exc:
            wire[0]={"ok":False,"error":type(exc).__name__+": "+str(exc)[:250]}
    dist.broadcast_object_list(wire,src=0)
    require(wire[0] and wire[0].get("ok"),"MANIFEST_"+str(wire[0]))
    dist.barrier()
    if rank==0:
        print("BL00_CONT_EPOCH_G4_PASS",completed,
              wire[0]["manifest_sha256"],flush=True)
    stop = completed==100 or (
        completed-trainer._bl00_session_first_epoch_index
        >= int(os.environ["BL00_MAX_SESSION_EPOCHS"])
    ) or (
        time.monotonic()-trainer._bl00_session_start
        >= float(os.environ["BL00_MAX_SESSION_SECONDS"])
    )
    status=[bool(stop) if rank==0 else None]
    dist.broadcast_object_list(status,src=0)
    if status[0]:
        raise BL00ContinuationSessionComplete(
            f"BL00_COMPLETED_{completed}_EPOCHS_BOUNDARY_STOP"
        )
'''

MODULES["bl00_continue_worker.py"] = r'''
"""Authenticated continuation from saved BL00 G4; no fresh starts, no B-TEST."""
from __future__ import annotations
import hashlib
import json
import os
import platform
import subprocess
from pathlib import Path
import torch


def need(ok,tag):
    if not ok:
        raise RuntimeError("BL00_CONT_WORKER_"+tag)


def sha(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for part in iter(lambda:f.read(1<<20),b""):
            h.update(part)
    return h.hexdigest()


def main():
    rank=int(os.environ["RANK"])
    local=int(os.environ["LOCAL_RANK"])
    need((rank,local,int(os.environ["WORLD_SIZE"])) in
         ((0,0,2),(1,1,2)), "RANK_PAIR")
    need(torch.cuda.device_count()==2 and all(
         "T4" in torch.cuda.get_device_name(i).upper() for i in (0,1)),
         "TWO_T4")
    torch.cuda.set_device(local)
    work=Path(os.environ["BL00_CONT_OUTPUT"])
    boot=Path(os.environ["BL00_CONT_BOOTSTRAP"])
    checkout=Path(os.environ["BL00_CHECKOUT"])
    private=Path(os.environ["BL00_PRIVATE_SOURCE"])
    need(subprocess.check_output(
        ["git","-C",str(checkout),"rev-parse","HEAD"],
        text=True).strip()==
        "f440d329d4544ec08ad63c1972bcf9a8b8a7eed8",
        "FROZEN_COMMIT")
    need(sha(private/"ultralytics/engine/trainer.py")==
         os.environ["BL00_PRIVATE_ENGINE_SHA256"],"PRIVATE_ENGINE_SHA")
    source_receipt=json.loads((work/"SOURCE_RECEIPT.json").read_text("utf-8"))
    for filename, expected in source_receipt["new_modules_sha256"].items():
        need(sha(boot/filename)==expected,
             "CONTINUATION_MODULE_SHA_"+filename)
    import ultralytics
    need(Path(ultralytics.__file__).resolve()==
         (private/"ultralytics/__init__.py").resolve(),
         "FROZEN_PRIVATE_IMPORT")
    from bl00_gate3_contract import RECIPE, validate_gate2b_receipt
    from bl00_continue_trainer import BL00ContinueTrainer
    from bl00_continue_g4 import BL00ContinuationSessionComplete
    previous=Path(os.environ["BL00_PRIOR_G4"])
    manifest_path=previous/os.environ["BL00_PRIOR_MANIFEST_FILE"]
    need(sha(manifest_path)==os.environ["BL00_PRIOR_MANIFEST_SHA"],
         "ORIGINAL_MANIFEST_SHA")
    manifest=json.loads(manifest_path.read_text("utf-8"))
    need(manifest["arm"]=="BL00" and manifest["complete"] is True and
         1<=manifest["completed_epochs"]<100 and
         manifest["b_test_access"]=="NONE","G4_METADATA")
    for name,digest in manifest["files"].items():
        need(name==Path(name).name and sha(previous/name)==digest,
             "G4_MEMBER_"+name)
    validate_gate2b_receipt(
        boot/"evidence/BL00_GATE2B_REAL_MODEL_RECEIPT.json"
    )
    args=dict(RECIPE)
    args.update({
        "model":str(checkout/
            "research/r9_training/frozen_source/ultralytics/cfg/models/11/yolo11.yaml"),
        "data":str(work/"BL00_CONT_train_val_only.yaml"),
        "project":str(work/"BL00_NATIVE_PROJECT"),
        "name":"BL00_CONT","mode":"train","pretrained":False,
        "close_mosaic":0,
    })
    need(args["epochs"]==100 and args["batch"]==16 and
         args["imgsz"]==1024 and args["workers"]==0 and
         args["optimizer"]=="SGD" and args["amp"] is True,"RECIPE")
    trainer=BL00ContinueTrainer(
        previous_g4=str(previous),previous_manifest=manifest,
        assignment_manifest=os.environ["BL00_ASSIGNMENT"],
        frozen_checkout=str(checkout),stock_yaml=args["model"],
        official_checkpoint=os.environ["BL00_PT"],
        gate2b_receipt=str(boot/"evidence/BL00_GATE2B_REAL_MODEL_RECEIPT.json"),
        initializer_module=str(boot/"bl00_initialization.py"),
        overrides=args,
    )
    need(trainer.world_size==2 and trainer.ddp is False,"NATIVE_TORCHRUN")
    try:
        trainer.train()
    except BL00ContinuationSessionComplete as exc:
        print("BL00_CONT_SESSION_BOUNDARY:",str(exc),flush=True)
    else:
        raise RuntimeError("BL00_CONT_NO_G4_BOUNDARY_STOP")
    latest=work/"g4/LATEST_MANIFEST.json"
    need(latest.is_file(),"NO_LATEST_MANIFEST")
    ptr=json.loads(latest.read_text("utf-8"))
    target=work/"g4"/ptr["manifest"]
    need(sha(target)==ptr["manifest_sha256"],"LATEST_MANIFEST_SHA")
    output=json.loads(target.read_text("utf-8"))
    need(output["complete"] is True
         and output["completed_epochs"]>manifest["completed_epochs"]
         and output["b_test_access"]=="NONE","INCOMPLETE_RUN")
    import torch.distributed as dist
    if dist.is_initialized():
        dist.barrier()
        dist.destroy_process_group()
    if rank==0:
        receipt={
            "schema":"BL00_CONTINUOUS_SESSION_RESULT_V1",
            "status":"PASS_RESUME_TRAIN_BVAL_G4",
            "starting_completed_epochs":manifest["completed_epochs"],
            "completed_epochs":output["completed_epochs"],
            "next_epoch_index":output["next_epoch_index"],
            "optimizer_updates":output["optimizer_updates"],
            "manifest_filename":ptr["manifest"],
            "manifest_sha256":ptr["manifest_sha256"],
            "best_fitness":output["best_fitness"],
            "val_map5095":output["val_metrics"]["metrics/mAP50-95(B)"],
            "b_test_access":"NONE",
            "source_manifest_sha256":os.environ["BL00_PRIOR_MANIFEST_SHA"],
            "torch":torch.__version__,
            "ultralytics":ultralytics.__version__,
            "python":platform.python_version(),
        }
        with (work/"BL00_CONTINUATION_RESULT.json").open("x") as f:
            json.dump(receipt,f,sort_keys=True,indent=2)
            f.write("\n")
        print("BL00_CONTINUATION_RESULT:",json.dumps(receipt,sort_keys=True),
              flush=True)


if __name__=="__main__":
    main()
'''

MODULES["bl00_continue_safe_load.py"] = r'''
"""Narrow NumPy allowlist for authenticated BL00 G4, preserving weights_only=True."""
from __future__ import annotations
from pathlib import Path
import hashlib
import numpy as np
import torch

ALLOWED_PICKLE_GLOBALS = {
    "numpy._core.multiarray.scalar",
    "numpy.core.multiarray.scalar",
    "numpy.dtype",
}

def load_authenticated_rank_state(path, expected_sha256):
    path = Path(path)
    if not path.is_file() or path.is_symlink():
        raise RuntimeError("BL00_CONT_G4_RANK_FILE_MISSING_OR_SYMLINK")
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != expected_sha256:
        raise RuntimeError("BL00_CONT_G4_RANK_SHA_MISMATCH")
    unexpected = set(torch.serialization.get_unsafe_globals_in_checkpoint(path)) - ALLOWED_PICKLE_GLOBALS
    if unexpected:
        raise RuntimeError("BL00_CONT_G4_UNREVIEWED_PICKLE_GLOBALS_" + repr(sorted(unexpected)))
    # Accepted Epoch-1 SGD state may include NumPy scalars from native np.interp.
    # Keep restricted checkpoint deserialization, never enable unrestricted pickle.
    scalar = np._core.multiarray.scalar
    safe = [
        (scalar, "numpy._core.multiarray.scalar"),
        (scalar, "numpy.core.multiarray.scalar"),
        np.dtype,
    ]
    safe.extend({
        type(np.dtype(t)) for t in (
            np.bool_, np.int8, np.int16, np.int32, np.int64,
            np.uint8, np.uint16, np.uint32, np.uint64,
            np.float16, np.float32, np.float64
        )
    })
    with torch.serialization.safe_globals(safe):
        state = torch.load(path, map_location="cpu", weights_only=True)
    if not isinstance(state, dict) or state.get("arm") != "BL00":
        raise RuntimeError("BL00_CONT_G4_RANK_SCHEMA_REFUSED")
    return state
'''

# -------- Authenticate the prior Kaggle source and G4 without GPU access --------
need(PRIOR_NOTEBOOK_OUTPUT.is_dir(), "PRIOR_NOTEBOOK_MOUNT_MISSING")
# Check the exact prior G4 before staging anything or launching GPUs.
expected_g4 = (
    PRIOR_NOTEBOOK_OUTPUT / "BL00_CONTINUOUS_B16_OUTPUT"
    / "g4" / "BL00_EPOCH033_two_rank_manifest.json"
)
need(safe_regular(expected_g4), "EPOCH33_SAVED_OUTPUT_NOT_ATTACHED")
need(
    sha256_file(expected_g4) == EXPECTED_PREVIOUS_MANIFEST_SHA256,
    "EPOCH33_MANIFEST_SHA_DRIFT",
)
candidate_roots = [
    p for p in PRIOR_NOTEBOOK_OUTPUT.rglob("BL00_EPOCH1_PORTABLE_BOOTSTRAP_R2")
    if p.is_dir() and not p.is_symlink()
]
# Later continuation outputs use BL00_CONTINUOUS_B16_OUTPUT/bootstrap.
candidate_roots.extend(
    p for p in PRIOR_NOTEBOOK_OUTPUT.rglob("bootstrap")
    if (p.is_dir() and not p.is_symlink()
        and p.parent.name == "BL00_CONTINUOUS_B16_OUTPUT"
        and (p/"bl00_epoch1_runner.py").is_file())
)
need(len(candidate_roots) == 1, "AUTHENTIC_PREVIOUS_BOOTSTRAP_NOT_UNIQUE")
prior_bootstrap = candidate_roots[0]
prior_runner = load_py(prior_bootstrap / "bl00_epoch1_runner.py",
                       "_bl00_prior_runner")
need(prior_runner.verify_members(prior_bootstrap) == 13,
     "PRIOR_BOOTSTRAP_AUTHENTICATION")
validator = (prior_bootstrap / "bl00_epoch1_validator.py").read_text("utf-8")
need("self.gather_stats()" not in validator
     and "self._d4_fixed_seen != 3049" in validator,
     "PRIOR_R2_VALIDATOR_NOT_FIXED")

manifest_candidates = []
for p in PRIOR_NOTEBOOK_OUTPUT.rglob("BL00_EPOCH*_two_rank_manifest.json"):
    if not safe_regular(p):
        continue
    try:
        data = json.loads(p.read_text("utf-8"))
        if (data.get("arm") != "BL00" or data.get("complete") is not True
                or data.get("frozen_commit") != FROZEN_COMMIT
                or data.get("b_test_access") != "NONE"):
            continue
        # Always verify every referenced member, not merely the manifest.
        if not all(
            name == Path(name).name
            and safe_regular(p.parent/name)
            and sha256_file(p.parent/name) == digest
            for name, digest in data.get("files",{}).items()
        ):
            continue
        manifest_candidates.append((int(data["completed_epochs"]),p,data))
    except (OSError, ValueError, KeyError, TypeError):
        continue
need(manifest_candidates, "NO_AUTHENTIC_G4_FOUND")
manifest_candidates.sort(key=lambda x:x[0],reverse=True)
completed, previous_manifest_path, prior_meta = manifest_candidates[0]
need(len(manifest_candidates)==1 or
     manifest_candidates[0][0]>manifest_candidates[1][0],
     "DUPLICATE_LATEST_G4")
previous_sha = sha256_file(previous_manifest_path)
need(completed == EXPECTED_PREVIOUS_COMPLETED_EPOCHS,
     "EPOCH33_NOT_LATEST_OR_WRONG_VERSION")
need(previous_manifest_path == expected_g4,
     "EPOCH33_G4_LOCATION_DRIFT")
need(previous_sha == EXPECTED_PREVIOUS_MANIFEST_SHA256,
     "EPOCH33_EXACT_SHA_DRIFT")
need(prior_meta["optimizer_updates"] == EXPECTED_PREVIOUS_OPTIMIZER_UPDATES,
     "EPOCH33_OPTIMIZER_CURSOR_DRIFT")
need(prior_meta["next_epoch_index"] == EXPECTED_PREVIOUS_COMPLETED_EPOCHS,
     "EPOCH33_NEXT_EPOCH_CURSOR_DRIFT")
if completed == 1:
    need(previous_sha == EPOCH1_MANIFEST_SHA256,
         "ORIGINAL_EPOCH1_SHA_DRIFT")
else:
    pointer=previous_manifest_path.parent/"LATEST_MANIFEST.json"
    need(pointer.is_file(),"LATEST_POINTER_MISSING")
    info=json.loads(pointer.read_text("utf-8"))
    need(info["manifest"]==previous_manifest_path.name and
         info["manifest_sha256"]==previous_sha and
         info["completed_epochs"]==completed,"LATEST_POINTER_INVALID")
need(completed<100,"ALREADY_COMPLETE_100")
need(all(k in prior_meta for k in (
    "ranks","files","best_fitness","optimizer_updates",
    "last_opt_step","accumulate","selected_best_sha256",
    "val_metrics"
)),"INCOMPLETE_G4_METADATA")
print("AUTHENTICATED G4:",previous_manifest_path)
print("COMPLETED EPOCHS:",completed,"NEXT:",completed+1)
print("RESTORE OPTIMIZER UPDATES:",prior_meta["optimizer_updates"])
print("G4 MANIFEST SHA256:",previous_sha)

# -------- Stage only authenticated portable sources and newly authored modules --------
need(not OUTPUT.exists(), "OUTPUT_ALREADY_EXISTS")
OUTPUT.mkdir(parents=True,exist_ok=False)
BOOTSTRAP.mkdir()
original_members = json.loads(
    (prior_bootstrap/"SOURCE_SHA256.json").read_text("utf-8")
)
for name,pin in original_members.items():
    source=prior_bootstrap/name
    need(safe_regular(source) and sha256_file(source)==pin,
         "SOURCE_SHA_"+name)
    target=BOOTSTRAP/name
    target.parent.mkdir(parents=True,exist_ok=True)
    shutil.copyfile(source,target)
# The accepted R2 manifest is retained unchanged as the predecessor identity.
shutil.copyfile(prior_bootstrap/"SOURCE_SHA256.json",
                BOOTSTRAP/"SOURCE_SHA256.json")

# Extend only the rank0 epoch restriction; R2's safe B-VAL body stays intact.
rank0_original=(BOOTSTRAP/"bl00_epoch1_rank0.py").read_text("utf-8")
need(rank0_original.count("epoch!=0")==1
     and rank0_original.count("getattr(trainer,'epoch',None)==0")==1,
     "RANK0_GUARD_ANCHORS")
MODULES["bl00_continue_rank0.py"]=rank0_original.replace(
    "epoch!=0","epoch<1",1
).replace(
    "getattr(trainer,'epoch',None)==0",
    "getattr(trainer,'epoch',None)==epoch",1
).replace("def validate_epoch1(","def validate_epoch(",1)

new_manifest={}
for name,src in MODULES.items():
    compile(src,name,"exec")
    target=BOOTSTRAP/name
    target.write_text(src,encoding="utf-8")
    new_manifest[name]=sha256_file(target)
print("ADDITIVE CONTINUATION MODULES:",len(new_manifest))

# CPU-only restricted checkpoint decode before GPU data scanning.
# Exposes unsupported serialization types without redoing training scans.
preflight_loader = load_py(
    BOOTSTRAP / "bl00_continue_safe_load.py",
    "_bl00_rank_loader_cpu_preflight",
)
for _rank in (0, 1):
    _row = prior_meta["ranks"][str(_rank)]
    _path = previous_manifest_path.parent / _row["filename"]
    _state = preflight_loader.load_authenticated_rank_state(_path, _row["sha256"])
    need(
        _state.get("rank") == _rank
        and _state.get("world_size") == 2
        and _state.get("optimizer_updates") == prior_meta["optimizer_updates"],
        "CPU_RANK_PRECHECK_" + str(_rank),
    )
    del _state
print("BL00_RESTRICTED_G4_DESERIALIZATION: TWO_RANK_CPU_PREFLIGHT_PASS",flush=True)

# -------- Verify frozen source and the exact scientific recipe --------
sys.path.insert(0,str(BOOTSTRAP))
from bl00_epoch1_private_patch import patch
from bl00_gate3_contract import RECIPE, NAME_ORDER, validate_recipe
need(prior_runner.regular(prior_runner.BUNDLE)
     and prior_runner.sha(prior_runner.BUNDLE)==prior_runner.BUNDLE_SHA,
     "FROZEN_BUNDLE_SHA")
pt=prior_runner.discover_pt()
assignment=prior_runner.discover_assignment()
split=prior_runner.SPLIT_DATA
train=split/"images/train"
val=split/"images/val"
need(all(p.is_dir() for p in (
    train,val,split/"labels/train",split/"labels/val"
)),"SPLIT_B_TRAIN_VAL")
import torch
need(torch.cuda.device_count()==2
     and all("T4" in torch.cuda.get_device_name(i).upper()
             for i in (0,1)),"TWO_T4_REQUIRED")
checkout=OUTPUT/"checkout"
cp=subprocess.run(
    ["git","clone","--quiet","--no-checkout",
     str(prior_runner.BUNDLE),str(checkout)],
    capture_output=True,text=True,timeout=240
)
need(cp.returncode==0,"GIT_CLONE_"+cp.stderr[-150:])
cp=subprocess.run(
    ["git","-C",str(checkout),"checkout","--quiet","--detach",FROZEN_COMMIT],
    capture_output=True,text=True,timeout=240
)
need(cp.returncode==0,"PINNED_CHECKOUT_"+cp.stderr[-150:])
need(subprocess.check_output(
    ["git","-C",str(checkout),"rev-parse","HEAD"],
    text=True).strip()==FROZEN_COMMIT,"HEAD_DRIFT")

source=checkout/"research/r9_training/frozen_source"
private=OUTPUT/"private_scientific_source"
shutil.copytree(source,private,symlinks=False)
original_engine=(source/"ultralytics/engine/trainer.py").read_bytes()
patched_engine=patch(original_engine).decode("utf-8")
replacements=[
    ("from bl00_epoch1_rank0 import validate_epoch1",
     "from bl00_continue_rank0 import validate_epoch"),
    ("validate_epoch1(self, epoch)","validate_epoch(self, epoch)"),
    ("from bl00_epoch1_g4 import capture_epoch1_and_stop",
     "from bl00_continue_g4 import capture_and_maybe_stop"),
    ("capture_epoch1_and_stop(self, epoch, nb, last_opt_step)",
     "capture_and_maybe_stop(self, epoch, nb, last_opt_step)"),
    ("last_opt_step = -1\n",
     "last_opt_step = self._bl00_resume_last_opt_step\n"),
    ('if not getattr(self, "_d4e_recovery_restored", False):\n'
     '            self.optimizer.zero_grad()',
     'if not getattr(self, "_bl00_resume_loaded", False):\n'
     '            self.optimizer.zero_grad()'),
]
for old,new in replacements:
    need(patched_engine.count(old)==1,
         "PRIVATE_ENGINE_ANCHOR_"+old[:35])
    patched_engine=patched_engine.replace(old,new,1)
ast.parse(patched_engine,filename="private/ultralytics/engine/trainer.py")
private_engine=private/"ultralytics/engine/trainer.py"
private_engine.write_text(patched_engine,encoding="utf-8")
need((source/"ultralytics/engine/trainer.py").read_bytes()==original_engine,
     "FROZEN_ENGINE_MODIFIED")

import yaml
train_val={"train":str(train),"val":str(val),
           "nc":9,"names":list(NAME_ORDER)}
data_yaml=OUTPUT/"BL00_CONT_train_val_only.yaml"
with data_yaml.open("x",encoding="utf-8") as f:
    yaml.safe_dump(train_val,f,sort_keys=False)
model_yaml=checkout/"research/r9_training/frozen_source/ultralytics/cfg/models/11/yolo11.yaml"
args=dict(RECIPE)
args.update({
    "model":str(model_yaml),"data":str(data_yaml),
    "project":str(OUTPUT/"BL00_NATIVE_PROJECT"),
    "name":"BL00_CONT","mode":"train","pretrained":False,
    "close_mosaic":0
})
validate_recipe(args)
(OUTPUT/"g4").mkdir()
(OUTPUT/"yolo_config").mkdir()
source_receipt={
    "schema":"BL00_CONTINUATION_SOURCE_RECEIPT_V1",
    "previous_g4_sha256":previous_sha,
    "previous_completed_epochs":completed,
    "frozen_commit":FROZEN_COMMIT,
    "private_engine_sha256":sha256_file(private_engine),
    "new_modules_sha256":new_manifest,
    "output":str(OUTPUT),
    "b_test_access":"NONE",
}
(OUTPUT/"SOURCE_RECEIPT.json").write_text(
    json.dumps(source_receipt,sort_keys=True,indent=2)+"\n",
    encoding="utf-8"
)

# -------- Launch: strict restoration of both Epoch-33 rank states precedes Epoch 34 --------
environment=dict(os.environ)
environment.update({
    "CUDA_VISIBLE_DEVICES":"0,1",
    "YOLO_OFFLINE":"true",
    "PYTHONDONTWRITEBYTECODE":"1","PYTHONHASHSEED":"42",
    "OMP_NUM_THREADS":"1","MKL_NUM_THREADS":"1",
    "YOLO_CONFIG_DIR":str(OUTPUT/"yolo_config"),
    "PYTHONPATH":os.pathsep.join(
        (str(private),str(checkout),str(BOOTSTRAP))
    ),
    "BL00_CONT_OUTPUT":str(OUTPUT),
    "BL00_CONT_BOOTSTRAP":str(BOOTSTRAP),
    "BL00_PRIVATE_SOURCE":str(private),
    "BL00_CHECKOUT":str(checkout),
    "BL00_PRIVATE_ENGINE_SHA256":sha256_file(private_engine),
    "BL00_PT":str(pt),"BL00_ASSIGNMENT":str(assignment),
    "BL00_PRIOR_G4":str(previous_manifest_path.parent),
    "BL00_PRIOR_MANIFEST_FILE":previous_manifest_path.name,
    "BL00_PRIOR_MANIFEST_SHA":previous_sha,
    "BL00_MAX_SESSION_EPOCHS":str(MAX_EPOCHS_THIS_SESSION),
    "BL00_MAX_SESSION_SECONDS":str(MAX_WALL_HOURS_THIS_SESSION*3600),
})

print("="*72)
print("BL00 EXACT G4 RESTORE + CONTINUOUS TRAINING",flush=True)
print("SOURCE:",previous_manifest_path,flush=True)
print("EPOCHS:",completed+1,"through 100 (bounded per session)",flush=True)
print("THIS SESSION: max",MAX_EPOCHS_THIS_SESSION,"epochs or",
      MAX_WALL_HOURS_THIS_SESSION,"hours",flush=True)
print("B-TEST: SEALED",flush=True)
print("="*72)

cmd=[sys.executable,"-m","torch.distributed.run",
     "--standalone","--nproc_per_node=2",
     str(BOOTSTRAP/"bl00_continue_worker.py")]
stdout=OUTPUT/"WORKER_STDOUT.log"
stderr=OUTPUT/"WORKER_STDERR.log"
with stdout.open("x",encoding="utf-8") as so, stderr.open("x",encoding="utf-8") as se:
    process=subprocess.Popen(cmd,env=environment,cwd=OUTPUT,
                             stdout=so,stderr=se)
    next_report=time.monotonic()+180
    while process.poll() is None:
        time.sleep(8)
        if time.monotonic()<next_report:
            continue
        next_report=time.monotonic()+180
        for path,label in ((stdout,"STDOUT"),(stderr,"STDERR")):
            try:
                with path.open("rb") as f:
                    f.seek(max(0,path.stat().st_size-2200))
                    tail=f.read().decode("utf-8",errors="replace")
                rows=[x.strip() for x in tail.replace("\r","\n").split("\n")
                      if x.strip()]
                print("BL00_PROGRESS_"+label+":",
                      rows[-1][-280:] if rows else "no output",flush=True)
            except OSError:
                pass

need(process.returncode==0,
     "WORKER_FAILED_"+
     stderr.read_text("utf-8",errors="replace")[-1500:])
receipt=OUTPUT/"BL00_CONTINUATION_RESULT.json"
need(safe_regular(receipt),"RESULT_MISSING")
result=json.loads(receipt.read_text("utf-8"))
latest=json.loads((OUTPUT/"g4/LATEST_MANIFEST.json").read_text("utf-8"))
latest_path=OUTPUT/"g4"/latest["manifest"]
need(sha256_file(latest_path)==latest["manifest_sha256"],
     "FINAL_MANIFEST_SHA")
final=json.loads(latest_path.read_text("utf-8"))
need(final["completed_epochs"]>completed
     and final["completed_epochs"]<=100
     and final["b_test_access"]=="NONE"
     and all(sha256_file(OUTPUT/"g4"/n)==d
             for n,d in final["files"].items()),"FINAL_G4_FILES")
print("="*72)
print("BL00 CONTINUATION SESSION: PASS",flush=True)
print("COMPLETED EPOCHS:",result["completed_epochs"],"/100",flush=True)
print("OPTIMIZER UPDATES:",result["optimizer_updates"],flush=True)
print("LAST B-VAL mAP50-95:",result["val_map5095"],flush=True)
print("LATEST G4 SHA256:",result["manifest_sha256"],flush=True)
print("LATEST G4:",latest_path,flush=True)
print("B-TEST: SEALED",flush=True)
print("="*72)
